# Coleta dos dados da API

In [628]:
import requests
import json

import pandas as pd
import numpy as np
import math

## Funções para Query

Aqui, definimos as funções para, efetivamente, fazer as requisições para a API da AniList.

Começamos definindo uma constante e uma função para facilitar a escrita das requisições.

In [629]:
URL = 'https://graphql.anilist.co'

In [630]:
def embedPageQuery(query:str):
    """Coloca uma query dentro da estrutura de requisição por páginas da AniList.
        Isso evita a reescrita do mesmo código"""
    
    embededQuery =  '''
                    query ($page: Int, $perPage: Int) {
                        page: Page (page: $page, perPage: $perPage){
                    '''
    embededQuery += query
    embededQuery += "}}"
    
    return embededQuery

Agora, vamos definiar as funções para solicitar os dados à API.

`queryPages` é a função onde efetivamente é realizada a requisição. A AniList estrurua a requisição de vários dados através de "páginas" (_pages_), as quais tem no máximo 50 entradas. Portanto, se quisermos mais, vamos ter que fazer várias requisições. Portanto, esse código faz várias requisições e concatena as respostas.

O `getPages`, por outro lado, considerada que os dados da AniList vão mudar muito pouco com o tempo, então para não precisar fazer a requisição toda vez que executar o código, os dados recebidos por `queryPages` são salvos no arquivo .json, caso ele ainda não exista. No caso do arquivo já existir, o código pega os dados diretos desse arquivos, sem nem fazer a requisação à API.

In [631]:
def queryPages(url:str, query:str, total:int=50):
    """Coleta <total> entradas da query <query> em <url>"""
    
    lasting = total
    
    pages = []
    while lasting > 0:

        variables = {
            'page': math.floor((total-lasting)/50)+1,
            'perPage': lasting
        }

        print("Requisitando dados...")
        response = requests.post(url, json={'query': embedPageQuery(query), 'variables': variables})
        pages.append(json.loads(response.content)["data"]['page'])

        lasting -= 50

    jointPages = []
    key = list(pages[0].keys())[0]
    for page in pages:
        jointPages += page[key]

    return jointPages

In [632]:
def getPages(url:str, query:str, name:str, total:int=50):
    """Busca pela query nos arquivos de cache. Se não encontrar, faz a requisição."""
    
    try:
        with open(f'queryCache/{name}.json', 'r') as file:
            data = json.load(file)
            
        if data['query'] != query or data['entryAmount'] != total:
            raise FileNotFoundError # Não é exatamente '_file_ not found', mas é perto o suficiente 
        
        pages = data['pages']
        print("Dados em cache!")

    except FileNotFoundError:
        pages = queryPages(url, query, total)

        data = {
            'pages': pages,
            'query': query,
            'entryAmount': total
        }

        with open(f'queryCache/{name}.json', 'w') as file:
            file.write(json.dumps(data, indent=4))

    return pages

## Requisições de Identidades

Definimos as queries a ser feitas

In [633]:
mainQuery = '''media (sort: SCORE_DESC) {
                    id
                    title {
                        romaji
                        english
                        native
                    }
                    coverImage {
                        large
                    }
                    type
                    format
                    genres
                    status
                    description
                    startDate {
                        year
                        month
                        day
                    }
                    endDate {
                        year
                        month
                        day
                    }

                    episodes
                    chapters
                    volumes

                    characters (sort: FAVOURITES_DESC, page: 1, perPage: 25) {
                        nodes{
                          id
                          name {
                             full
                          }
                          image {
                              large
                          }
                          description
                          gender
                          age
                        }
                    }

                    studios (sort: FAVOURITES_DESC) {
                        nodes{
                          id
                          name
                        }
                        edges {
                          isMain
                        }
                    }

                    staff (sort: FAVOURITES_DESC, page: 1, perPage: 25) {
                      nodes{
                        id
                        image {
                            large
                        }
                        name {
                            full
                        }
                        description
                        gender
                        age
                        primaryOccupations
                        languageV2
                      }
                      edges{
                        role
                      }
                    }
                }
            '''

In [634]:
userQuery = '''users (sort: WATCHED_TIME){
                    id
                    name
                    avatar {
                        large
                    }
                    createdAt
                }
            '''

Organizamos as queries a serem feitas junto ao nome do seu arquivo e a quantidade de elementos para aquela query

In [635]:
queries = {
    'main':    (mainQuery,    500),
    'user':     (userQuery,     20)
}

Fazermos a requisição

In [636]:
dfs = {}
for name, (query, total) in queries.items():
    print(f"\n-- {name.capitalize()} --")
    pages = getPages(URL, query, name, total)
    dfs[name] = pd.DataFrame(pages)


-- Main --
Dados em cache!

-- User --
Dados em cache!


## Retorno das queries

In [637]:
dfs['main']

,id,title,coverImage,type,format,genres,status,description,startDate,endDate,episodes,chapters,volumes,characters,studios,staff
0,30002,"{'romaji': 'Berserk', 'english': 'Berserk', 'n...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Drama, Fantasy, Horror, Ps...",RELEASING,"His name is Guts, the Black Swordsman, a feare...","{'year': 1989, 'month': 8, 'day': 25}","{'year': None, 'month': None, 'day': None}",NaN,NaN,NaN,"{'nodes': [{'id': 422, 'name': {'full': 'Guts'...","{'nodes': [], 'edges': []}","{'nodes': [{'id': 96868, 'image': {'large': 'h..."
1,31706,{'romaji': 'JoJo no Kimyou na Bouken: Steel Ba...,{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Drama, Mystery, Supernatural]",FINISHED,Riders from around the world gather in the Wil...,"{'year': 2004, 'month': 1, 'day': 19}","{'year': 2011, 'month': 4, 'day': 19}",NaN,95.0,24.0,"{'nodes': [{'id': 11772, 'name': {'full': 'Gyr...","{'nodes': [], 'edges': []}","{'nodes': [{'id': 97619, 'image': {'large': 'h..."
2,30656,"{'romaji': 'Vagabond', 'english': 'Vagabond', ...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Drama]",RELEASING,"At seventeen years of age, Miyamoto Musashi--s...","{'year': 1998, 'month': 9, 'day': 3}","{'year': None, 'month': None, 'day': None}",NaN,NaN,NaN,"{'nodes': [{'id': 6194, 'name': {'full': 'Musa...","{'nodes': [], 'edges': []}","{'nodes': [{'id': 96911, 'image': {'large': 'h..."
3,30013,"{'romaji': 'ONE PIECE', 'english': 'One Piece'...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Comedy, Fantasy]",RELEASING,"As a child, Monkey D. Luffy was inspired to be...","{'year': 1997, 'month': 7, 'day': 22}","{'year': None, 'month': None, 'day': None}",NaN,NaN,NaN,"{'nodes': [{'id': 40, 'name': {'full': 'Luffy ...","{'nodes': [], 'edges': []}","{'nodes': [{'id': 96881, 'image': {'large': 'h..."
4,154587,"{'romaji': 'Sousou no Frieren', 'english': 'Fr...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Adventure, Drama, Fantasy]",FINISHED,The adventure is over but life goes on for an ...,"{'year': 2023, 'month': 9, 'day': 29}","{'year': 2024, 'month': 3, 'day': 22}",28.0,NaN,NaN,"{'nodes': [{'id': 176754, 'name': {'full': 'Fr...","{'nodes': [{'id': 11, 'name': 'MADHOUSE'}, {'i...","{'nodes': [{'id': 194287, 'image': {'large': '..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,35249,"{'romaji': 'NHK ni Youkoso!', 'english': 'Welc...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,NOVEL,"[Comedy, Drama, Psychological, Slice of Life]",FINISHED,"Tatsuhiro Satou, a university dropout entering...","{'year': 2002, 'month': 1, 'day': 28}","{'year': 2002, 'month': 1, 'day': 28}",NaN,12.0,1.0,"{'nodes': [{'id': 2552, 'name': {'full': 'Misa...","{'nodes': [], 'edges': []}","{'nodes': [{'id': 96991, 'image': {'large': 'h..."
496,36812,"{'romaji': 'Kyou kara Ore wa!!', 'english': No...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,[Comedy],FINISHED,After Takashi's family moves into a new neighb...,"{'year': 1988, 'month': 2, 'day': 10}","{'year': 1997, 'month': 11, 'day': 15}",NaN,366.0,38.0,"{'nodes': [{'id': 18367, 'name': {'full': 'Tak...","{'nodes': [], 'edges': []}","{'nodes': [{'id': 97770, 'image': {'large': 'h..."
497,41327,"{'romaji': 'One Outs', 'english': None, 'nativ...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Psychological, Sports]",FINISHED,"The story begins when Hiromichi Kojima, the st...","{'year': 1998, 'month': 6, 'day': 18}","{'year': 2009, 'month': 2, 'day': 19}",NaN,176.0,20.0,"{'nodes': [{'id': 16589, 'name': {'full': 'Tou...","{'nodes': [], 'edges': []}","{'nodes': [{'id': 97656, 'image': {'large': 'h..."
498,86406,"{'romaji': 'Kumo desu ga, Nani ka?', 'english'...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,NOVEL,"[Adventure, Comedy, Fantasy]",FINISHED,I used to be a normal high school girl but i

In [638]:
keys = [('characters', 'character'), ('studios', 'studio'), ('staff', 'staff')]

for (oldKey, newKey) in keys:
    temp = dfs['main'][oldKey].apply(lambda x: pd.DataFrame(x['nodes']))
    temp = pd.concat(temp.tolist())
    dfs[newKey] = temp

In [639]:
dfs['character']

,id,name,image,description,gender,age
0,422,{'full': 'Guts'},{'large': 'https://s4.anilist.co/file/anilistc...,Guts is the protagonist of Berserk. He is a ta...,Male,None
1,424,{'full': 'Griffith'},{'large': 'https://s4.anilist.co/file/anilistc...,Griffith was the founder and leader of the mer...,Male,None
2,423,{'full': 'Casca'},{'large': 'https://s4.anilist.co/file/anilistc...,Casca was the only female soldier in the origi...,Female,None
3,5060,{'full': 'Puck'},{'large': 'https://s4.anilist.co/file/anilistc...,"A wind spirit from Elfhelm, a utopia on the is...",NaN,None
4,7857,{'full': 'Farnese de Vandimion'},{'large': 'https://s4.anilist.co/file/anilistc...,Farnese is a noblewoman who led the Holy See's...,Female,None
...,...,...,...,...,...,...
7,195667,{'full': 'Iori Hashima'},{'large': 'https://s4.anilist.co/file/anilistc...,Iori Hashima is the older brother of Izumi Has...,Male,NaN
8,195669,{'full': 'Echika Mizuhara'},{'large': 'https://s4.anilist.co/file/anilistc...,Echika Mizuhara is another oboist of the Icy t...,Female,NaN
9,195666,{'full': 'Sonoko Machida'},{'large': 'https://s4.anilist.co/file/anilistc...,Sonoko Machida works at Fujikawa Shoten and is...,Female,NaN
10,195670,{'full': 'Tokino Himekawa'},{'large': 'https://s4.anilist.co/file/anilistc...,Tokino Himekawa is another oboist of the Icy t...,Female,NaN


In [640]:
# bad and stinky but it works so let's roll with it
for (oldKey, newKey) in keys:
    relation = f'{newKey}_midia'
    dfs[relation] = pd.DataFrame(columns=['id_media', f'id_{newKey}'])

    for (i, row) in dfs['main'].iterrows():
        subDfs = pd.DataFrame(row[oldKey]['nodes'])
        for (j, subdf) in subDfs.iterrows():
            dfs[relation].loc[len(dfs[relation])] =  [row['id'], subdf['id']]

In [641]:
dfs['character_midia']

,id_media,id_character
0,30002,422
1,30002,424
2,30002,423
3,30002,5060
4,30002,7857
...,...,...
9807,100675,195667
9808,100675,195669
9809,100675,195666
9810,100675,195670


In [642]:
dfs['studio_midia']

,id_media,id_studio
0,154587,11
1,154587,17
2,154587,245
3,154587,53
4,154587,62
...,...,...
1140,20799,415
1141,20799,7079
1142,20811,858
1143,20811,10


In [643]:
dfs['studio_midia']['isMain'] = False
i=0
for (i, row) in dfs['main'].iterrows():
        subDfs = pd.DataFrame(row['studios']['edges'])
        for (j, subdf) in subDfs.iterrows():
            if subdf.empty:
                  break
            dfs['studio_midia'].loc[i, 'isMain'] = subdf['isMain']
            i+=1

In [644]:
dfs['studio_midia']

,id_media,id_studio,isMain
0,154587,11,False
1,154587,17,False
2,154587,245,False
3,154587,53,False
4,154587,62,True
...,...,...,...
1140,20799,415,False
1141,20799,7079,False
1142,20811,858,False
1143,20811,10,False


In [646]:
dfs['staff_midia']['role'] = None
i=0
for (i, row) in dfs['main'].iterrows():
        subDfs = pd.DataFrame(row['staff']['edges'])
        for (j, subdf) in subDfs.iterrows():
            if subdf.empty:
                  break
            dfs['staff_midia'].loc[i, 'role'] = subdf['role']
            i+=1

In [647]:
dfs['staff_midia']

,id_media,id_staff,role
0,30002,96868,Story & Art (vols 1-41)
1,30002,101748,Story & Art
2,30002,220675,Story & Art
3,30002,143994,Story & Art
4,30002,219478,Theme Song Performance (OP1)
...,...,...,...
7694,100675,133511,None
7695,100675,160052,None
7696,100675,147288,None
7697,100675,124718,None


In [ ]:
dfs['staff']

,id,image,name,description,gender,age,primaryOccupations,languageV2
0,96868,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kentarou Miura'},[Wiki (jp)](https://ja.wikipedia.org/wiki/%E4%...,Male,54.0,[Mangaka],Japanese
1,101748,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kouji Mori'},[Twitter](https://twitter.com/morichankenchan)...,Male,59.0,[Mangaka],Japanese
2,220675,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Drik Sada'},[Twitter](https://twitter.com/driksada),NaN,NaN,[Translator],Portuguese
3,143994,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Studio Gaga'},Studio Gaga is a Japanese manga production stu...,NaN,NaN,[],Japanese
4,219478,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Paweł Dybała'},[Twitter](https://twitter.com/pawel_dybala) <b...,Male,45.0,[Translator],Polish
...,...,...,...,...,...,...,...,...
20,133511,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kazumasa Ishida'},NaN,Male,NaN,[Animator],Japanese
21,160052,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Masaharu Yamanouchi'},Masaharu Yamanouchi is a music producer.,Male,NaN,[Producer],Japanese
22,147288,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Youko Matsuzaki'},Youko Matsuzaki is a producer at Fuji TV.,Female,NaN,[Producer],Japanese
23,124718,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Yoshiko Okuda'},NaN,Female,NaN,[Animator],Japanese


In [ ]:
midia = dfs['main']
del midia['characters']
del midia['studios']
del midia['staff']

dfs['anime'] = dfs['main'][dfs['main']['type'] == 'ANIME']
dfs['manga'] = dfs['main'][dfs['main']['type'] == 'MANGA']

del dfs['anime']['volumes']
del dfs['anime']['chapters']
del dfs['manga']['episodes']

dfs['anime']

,id,title,coverImage,type,format,genres,status,description,startDate,endDate,episodes
4,154587,"{'romaji': 'Sousou no Frieren', 'english': 'Fr...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Adventure, Drama, Fantasy]",FINISHED,The adventure is over but life goes on for an ...,"{'year': 2023, 'month': 9, 'day': 29}","{'year': 2024, 'month': 3, 'day': 22}",28.0
6,114129,"{'romaji': 'Gintama: THE FINAL', 'english': 'G...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Action, Comedy, Drama, Sci-Fi]",FINISHED,Two years have passed following the Tendoshuu'...,"{'year': 2021, 'month': 1, 'day': 8}","{'year': 2021, 'month': 1, 'day': 8}",1.0
7,20996,"{'romaji': 'Gintama°', 'english': 'Gintama Sea...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Comedy, Drama, Sci-Fi]",FINISHED,"Gintoki, Shinpachi, and Kagura return as the f...","{'year': 2015, 'month': 4, 'day': 8}","{'year': 2016, 'month': 3, 'day': 30}",51.0
10,189046,{'romaji': 'Re:Zero kara Hajimeru Isekai Seika...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Drama, Fantasy, Psychologi...",FINISHED,"In the deadly battle at Priestella, Subaru and...","{'year': 2026, 'month': 4, 'day': 8}","{'year': 2026, 'month': 9, 'day': 30}",19.0
12,171627,"{'romaji': 'Chainsaw Man: Reze-hen', 'english'...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Action, Drama, Horror, Romance, Supernatural]",FINISHED,"Denji became “Chainsaw Man”, a boy with a devi...","{'year': 2025, 'month': 9, 'day': 19}","{'year': 2025, 'month': 9, 'day': 19}",1.0
...,...,...,...,...,...,...,...,...,...,...,...
491,199221,"{'romaji': 'Dr. STONE: SCIENCE FUTURE Part 3',...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Comedy, Sci-Fi]",FINISHED,"After reconciling with Dr. Xeno, Senkuu Ishiga...","{'year': 2026, 'month': 4, 'day': 2}","{'year': 2026, 'month': 6, 'day': 25}",13.0
492,392,"{'romaji': 'Yuu☆Yuu☆Hakusho', 'english': 'Yu Y...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Comedy, Supernatural]",FINISHED,"Urameshi Yusuke, 14 years of age, is about as ...","{'year': 1992, 'month': 10, 'day': 10}","{'year': 1995, 'month': 1, 'day': 7}",112.0
493,20799,{'romaji': 'JoJo no Kimyou na Bouken: Stardust...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Drama, Supernatural]",FINISHED,"After 30 days of travel across the world, plag...","{'year': 2015, 'month': 1, 'day': 11}","{'year': 2015, 'month': 6, 'day': 20}",24.0
494,20811,{'romaji': 'Shingeki no Kyojin Gaiden: Kuinaki...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,OVA,"[Action, Drama, Fantasy]",FINISHED,This prequel to megahit Attack on Titan answer...,"{'year': 2014, 'month': 12, 'day': 9}","{'year': 2015, 'month': 4, 'day': 9}",2.0


In [ ]:
dfs['user']

,id,name,avatar,createdAt
0,6,rickywolfy,{'large': 'https://s4.anilist.co/file/anilistc...,1378575508
1,14,user14,{'large': 'https://s4.anilist.co/file/anilistc...,1397079561
2,16,Vaskur,{'large': 'https://s4.anilist.co/file/anilistc...,1397079561
3,18,user18,{'large': 'https://s4.anilist.co/file/anilistc...,1381929801
4,20,VectorPrime,{'large': 'https://s4.anilist.co/file/anilistc...,1397079561
5,23,JPFlySolo,{'large': 'https://s4.anilist.co/file/anilistc...,1398323844
6,24,1e90ff,{'large': 'https://s4.anilist.co/file/anilistc...,1398323844
7,26,thejag,{'large': 'https://s4.anilist.co/file/anilistc...,1397079561
8,31,DeadCake,{'large': 'https://s4.anilist.co/file/anilistc...,1397079561
9,41,MsNewwonders,{'large': 'https://s4.anilist.co/file/anilistc...,1397079561


## Expansão de colunas compostas

Na resposta da API, muitas colunas são compostas, sendo seus valores listas ou dicionários. Vamos expandí-las em tabelas próprias de acordo com o modelo lógico.

### Títulos

In [ ]:
dfs['title'] = pd.DataFrame()

animeTitles = pd.DataFrame()
mangaTitles = pd.DataFrame()

animeTitles['id_midia'] = dfs['anime']['id']
mangaTitles['id_midia'] = dfs['anime']['id']

titleTypes = ['romaji', 'english', 'native']

for titleType in titleTypes:
    animeTitles[titleType] = dfs['anime']['title'].apply(lambda x: x[titleType])
    mangaTitles[titleType] = dfs['manga']['title'].apply(lambda x: x[titleType])


dfs['title'] = pd.concat([animeTitles, mangaTitles])

dfs['title']

,id_midia,romaji,english,native
4,154587,Sousou no Frieren,Frieren: Beyond Journey’s End,葬送のフリーレン
6,114129,Gintama: THE FINAL,Gintama: THE VERY FINAL,銀魂 THE FINAL
7,20996,Gintama°,Gintama Season 3,銀魂゜
10,189046,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,Re:ZERO -Starting Life in Another World- Season 4,Re:ゼロから始める異世界生活 4th season
12,171627,Chainsaw Man: Reze-hen,Chainsaw Man – The Movie: Reze Arc,チェンソーマン レゼ篇
...,...,...,...,...
491,199221,NaN,NaN,NaN
492,392,NaN,NaN,NaN
493,20799,NaN,NaN,NaN
494,20811,NaN,NaN,NaN


In [ ]:
del dfs['anime']['title']
del dfs['manga']['title']

### Gêneros

In [ ]:
compondGenres = pd.concat([dfs['anime'][['id', 'genres']], dfs['manga'][['id', 'genres']]])

dfs['genre'] = pd.DataFrame(columns=['id_midia', 'name'])

# bad and stinky code but god has punished me for my hubris and my work is never finished 
# so I don't have the time to fix it
for id, occupations in compondGenres.values:
    for genre in occupations:
        dfs['genre'].loc[len(dfs['genre'])] = [id, genre]

dfs['genre']

,id_midia,name
0,154587,Adventure
1,154587,Drama
2,154587,Fantasy
3,114129,Action
4,114129,Comedy
...,...,...
1846,41327,Psychological
1847,41327,Sports
1848,86406,Adventure
1849,86406,Comedy


In [ ]:
del dfs['anime']['genres']
del dfs['manga']['genres']

### Ocupações

In [ ]:
compondOccupations = dfs['staff'][['id', 'primaryOccupations']]

dfs['occupation'] = pd.DataFrame(columns=['id_staff', 'occupation'])

# bad and stinky code but god has punished me for my hubris and my work is never finished 
# so I don't have the time to fix it
for id, occupations in compondOccupations.values:
    for occupation in occupations:
        dfs['occupation'].loc[len(dfs['occupation'])] = [id, occupation]

dfs['occupation']

,id_staff,occupation
0,96868,Mangaka
1,101748,Mangaka
2,220675,Translator
3,219478,Translator
4,214718,Translator
...,...,...
10596,133511,Animator
10597,160052,Producer
10598,147288,Producer
10599,124718,Animator


In [ ]:
partialStaff = dfs['staff']
partialStaff['id_staff'] = partialStaff['id']
mergedStaff = pd.merge(dfs['occupation'], partialStaff, on='id_staff')

voiceActors = mergedStaff[mergedStaff['occupation'] == 'Voice Actor']

dfs['va']    = voiceActors[['id_staff', 'languageV2']]
dfs['va']

,id_staff,languageV2
605,95472,English
606,95472,English
607,95472,English
608,95472,English
609,95472,English
...,...,...
100490,158618,Portuguese
100491,158618,Portuguese
100492,158618,Portuguese
100493,158618,Portuguese


In [ ]:
del dfs['staff']['primaryOccupations']

### Campos atômicos dentro de listas e dicionários

#### Imagens

In [ ]:
extractImage = lambda x: x['large']

dfs['anime']['coverImage']  = dfs['anime']['coverImage'].apply(extractImage)
dfs['manga']['coverImage']  = dfs['manga']['coverImage'].apply(extractImage)
dfs['staff']['image']       = dfs['staff']['image'].apply(extractImage)
dfs['user']['avatar']       = dfs['user']['avatar'].apply(extractImage)
dfs['character']['image']   = dfs['character']['image'].apply(extractImage)

In [ ]:
dfs['character']['image']

0     https://s4.anilist.co/file/anilistcdn/characte...
1     https://s4.anilist.co/file/anilistcdn/characte...
2     https://s4.anilist.co/file/anilistcdn/characte...
3     https://s4.anilist.co/file/anilistcdn/characte...
4     https://s4.anilist.co/file/anilistcdn/characte...
                            ...                        
7     https://s4.anilist.co/file/anilistcdn/characte...
8     https://s4.anilist.co/file/anilistcdn/characte...
9     https://s4.anilist.co/file/anilistcdn/characte...
10    https://s4.anilist.co/file/anilistcdn/characte...
11    https://s4.anilist.co/file/anilistcdn/characte...
Name: image, Length: 9812, dtype: str

#### Nomes

In [ ]:
extractName = lambda x: x['full']

dfs['character']['name'] = dfs['character']['name'].apply(extractName)
dfs['staff']['name'] = dfs['staff']['name'].apply(extractName)

In [ ]:
dfs['staff']['name']

0          Kentarou Miura
1              Kouji Mori
2               Drik Sada
3             Studio Gaga
4            Paweł Dybała
             ...         
20        Kazumasa Ishida
21    Masaharu Yamanouchi
22        Youko Matsuzaki
23          Yoshiko Okuda
24         Keita Nagasaka
Name: name, Length: 7699, dtype: str

## Tipagem

Alguns tipos vieram inapropriados, como float quando deveria ser inteiro ou datas quebradas de formas estranhas. Vamos ajeitas isso!

### Datas

In [ ]:
def dict2timestamp(x):
    try:
        return pd.Timestamp(year=x['year'], month=x['month'], day=x['day'])
    except:
        return None

dfs['anime']['startDate'] = dfs['anime']['startDate'].apply(dict2timestamp)
dfs['anime']['endDate']   = dfs['anime']['endDate'].apply(dict2timestamp)

dfs['manga']['startDate'] = dfs['manga']['startDate'].apply(dict2timestamp)
dfs['manga']['endDate']   = dfs['manga']['endDate'].apply(dict2timestamp)

In [ ]:
dfs['anime']['startDate']

4     2023-09-29
6     2021-01-08
7     2015-04-08
10    2026-04-08
12    2025-09-19
         ...    
491   2026-04-02
492   1992-10-10
493   2015-01-11
494   2014-12-09
499   2019-10-26
Name: startDate, Length: 256, dtype: datetime64[us]

In [ ]:
dfs['user']['createdAt'] = pd.to_datetime(dfs['user']['createdAt'], unit='s')

In [ ]:
dfs['user']['createdAt']

0    2013-09-07 17:38:28
1    2014-04-09 21:39:21
2    2014-04-09 21:39:21
3    2013-10-16 13:23:21
4    2014-04-09 21:39:21
5    2014-04-24 07:17:24
6    2014-04-24 07:17:24
7    2014-04-09 21:39:21
8    2014-04-09 21:39:21
9    2014-04-09 21:39:21
10   2014-04-09 21:39:21
11   2014-04-09 21:39:21
12   2014-04-09 21:39:21
13   2013-11-12 21:38:58
14   2014-04-09 21:39:21
15   2015-07-18 17:41:04
16   2014-04-09 21:39:21
17   2014-04-09 21:39:21
18   2014-04-09 21:39:21
19   2013-09-07 18:33:55
Name: createdAt, dtype: datetime64[s]

### Numeros

In [ ]:
dfs['anime']['episodes'] = dfs['anime']['episodes'].astype('Int64')

In [ ]:
dfs['manga']['chapters'] = dfs['manga']['chapters'].astype('Int64')
dfs['manga']['volumes'] = dfs['manga']['chapters'].astype('Int64')

In [ ]:
dfs['staff']['age'] = dfs['staff']['age'].astype('Int64')

## Dados limpos

In [ ]:
dfs['anime']

,id,coverImage,type,format,status,description,startDate,endDate,episodes
4,154587,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,The adventure is over but life goes on for an ...,2023-09-29,2024-03-22,28
6,114129,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,MOVIE,FINISHED,Two years have passed following the Tendoshuu'...,2021-01-08,2021-01-08,1
7,20996,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"Gintoki, Shinpachi, and Kagura return as the f...",2015-04-08,2016-03-30,51
10,189046,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"In the deadly battle at Priestella, Subaru and...",2026-04-08,2026-09-30,19
12,171627,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,MOVIE,FINISHED,"Denji became “Chainsaw Man”, a boy with a devi...",2025-09-19,2025-09-19,1
...,...,...,...,...,...,...,...,...,...
491,199221,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"After reconciling with Dr. Xeno, Senkuu Ishiga...",2026-04-02,2026-06-25,13
492,392,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"Urameshi Yusuke, 14 years of age, is about as ...",1992-10-10,1995-01-07,112
493,20799,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"After 30 days of travel across the world, plag...",2015-01-11,2015-06-20,24
494,20811,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,OVA,FINISHED,This prequel to megahit Attack on Titan answer...,2014-12-09,2015-04-09,2


In [ ]:
dfs['manga']

,id,coverImage,type,format,status,description,startDate,endDate,chapters,volumes
0,30002,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,RELEASING,"His name is Guts, the Black Swordsman, a feare...",1989-08-25,NaT,<NA>,<NA>
1,31706,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,Riders from around the world gather in the Wil...,2004-01-19,2011-04-19,95,95
2,30656,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,RELEASING,"At seventeen years of age, Miyamoto Musashi--s...",1998-09-03,NaT,<NA>,<NA>
3,30013,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,RELEASING,"As a child, Monkey D. Luffy was inspired to be...",1997-07-22,NaT,<NA>,<NA>
5,64053,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,It's time once again for the annual family gat...,2012-01-21,2015-06-22,42,42
...,...,...,...,...,...,...,...,...,...,...
488,169272,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,RELEASING,When Arisu Yamabuki was all alone in bed at ni...,2023-09-20,NaT,<NA>,<NA>
495,35249,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,NOVEL,FINISHED,"Tatsuhiro Satou, a university dropout entering...",2002-01-28,2002-01-28,12,12
496,36812,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,After Takashi's family moves into a new neighb...,1988-02-10,1997-11-15,366,366
497,41327,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,"The story begins when Hiromichi Kojima, the st...",1998-06-18,2009-02-19,176,176


In [ ]:
dfs['studio']

,id,name
0,11,MADHOUSE
1,17,Aniplex
2,245,Toho
3,53,Dentsu
4,62,Shogakukan-Shueisha Productions
...,...,...
1,415,Warner Bros. Japan
2,7079,ONE MUSIC
0,858,WIT STUDIO
1,10,Production I.G


In [ ]:
dfs['character']

,id,name,image,description,gender,age
0,422,Guts,https://s4.anilist.co/file/anilistcdn/characte...,Guts is the protagonist of Berserk. He is a ta...,Male,None
1,424,Griffith,https://s4.anilist.co/file/anilistcdn/characte...,Griffith was the founder and leader of the mer...,Male,None
2,423,Casca,https://s4.anilist.co/file/anilistcdn/characte...,Casca was the only female soldier in the origi...,Female,None
3,5060,Puck,https://s4.anilist.co/file/anilistcdn/characte...,"A wind spirit from Elfhelm, a utopia on the is...",NaN,None
4,7857,Farnese de Vandimion,https://s4.anilist.co/file/anilistcdn/characte...,Farnese is a noblewoman who led the Holy See's...,Female,None
...,...,...,...,...,...,...
7,195667,Iori Hashima,https://s4.anilist.co/file/anilistcdn/characte...,Iori Hashima is the older brother of Izumi Has...,Male,NaN
8,195669,Echika Mizuhara,https://s4.anilist.co/file/anilistcdn/characte...,Echika Mizuhara is another oboist of the Icy t...,Female,NaN
9,195666,Sonoko Machida,https://s4.anilist.co/file/anilistcdn/characte...,Sonoko Machida works at Fujikawa Shoten and is...,Female,NaN
10,195670,Tokino Himekawa,https://s4.anilist.co/file/anilistcdn/characte...,Tokino Himekawa is another oboist of the Icy t...,Female,NaN


In [ ]:
dfs['staff']

,id,image,name,description,gender,age,languageV2,id_staff
0,96868,https://s4.anilist.co/file/anilistcdn/staff/la...,Kentarou Miura,[Wiki (jp)](https://ja.wikipedia.org/wiki/%E4%...,Male,54,Japanese,96868
1,101748,https://s4.anilist.co/file/anilistcdn/staff/la...,Kouji Mori,[Twitter](https://twitter.com/morichankenchan)...,Male,59,Japanese,101748
2,220675,https://s4.anilist.co/file/anilistcdn/staff/la...,Drik Sada,[Twitter](https://twitter.com/driksada),NaN,<NA>,Portuguese,220675
3,143994,https://s4.anilist.co/file/anilistcdn/staff/la...,Studio Gaga,Studio Gaga is a Japanese manga production stu...,NaN,<NA>,Japanese,143994
4,219478,https://s4.anilist.co/file/anilistcdn/staff/la...,Paweł Dybała,[Twitter](https://twitter.com/pawel_dybala) <b...,Male,45,Polish,219478
...,...,...,...,...,...,...,...,...
20,133511,https://s4.anilist.co/file/anilistcdn/staff/la...,Kazumasa Ishida,NaN,Male,<NA>,Japanese,133511
21,160052,https://s4.anilist.co/file/anilistcdn/staff/la...,Masaharu Yamanouchi,Masaharu Yamanouchi is a music producer.,Male,<NA>,Japanese,160052
22,147288,https://s4.anilist.co/file/anilistcdn/staff/la...,Youko Matsuzaki,Youko Matsuzaki is a producer at Fuji TV.,Female,<NA>,Japanese,147288
23,124718,https://s4.anilist.co/file/anilistcdn/staff/la...,Yoshiko Okuda,NaN,Female,<NA>,Japanese,124718


In [ ]:
dfs['title']

,id_midia,romaji,english,native
4,154587,Sousou no Frieren,Frieren: Beyond Journey’s End,葬送のフリーレン
6,114129,Gintama: THE FINAL,Gintama: THE VERY FINAL,銀魂 THE FINAL
7,20996,Gintama°,Gintama Season 3,銀魂゜
10,189046,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,Re:ZERO -Starting Life in Another World- Season 4,Re:ゼロから始める異世界生活 4th season
12,171627,Chainsaw Man: Reze-hen,Chainsaw Man – The Movie: Reze Arc,チェンソーマン レゼ篇
...,...,...,...,...
491,199221,NaN,NaN,NaN
492,392,NaN,NaN,NaN
493,20799,NaN,NaN,NaN
494,20811,NaN,NaN,NaN


In [ ]:
dfs['user']

,id,name,avatar,createdAt
0,6,rickywolfy,https://s4.anilist.co/file/anilistcdn/user/ava...,2013-09-07 17:38:28
1,14,user14,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-04-09 21:39:21
2,16,Vaskur,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-04-09 21:39:21
3,18,user18,https://s4.anilist.co/file/anilistcdn/user/ava...,2013-10-16 13:23:21
4,20,VectorPrime,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-04-09 21:39:21
5,23,JPFlySolo,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-04-24 07:17:24
6,24,1e90ff,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-04-24 07:17:24
7,26,thejag,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-04-09 21:39:21
8,31,DeadCake,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-04-09 21:39:21
9,41,MsNewwonders,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-04-09 21:39:21


In [ ]:
dfs['genre']

,id_midia,name
0,154587,Adventure
1,154587,Drama
2,154587,Fantasy
3,114129,Action
4,114129,Comedy
...,...,...
1846,41327,Psychological
1847,41327,Sports
1848,86406,Adventure
1849,86406,Comedy


In [ ]:
dfs['occupation']

,id_staff,occupation
0,96868,Mangaka
1,101748,Mangaka
2,220675,Translator
3,219478,Translator
4,214718,Translator
...,...,...
10596,133511,Animator
10597,160052,Producer
10598,147288,Producer
10599,124718,Animator


In [ ]:
dfs['va']

,id_staff,languageV2
605,95472,English
606,95472,English
607,95472,English
608,95472,English
609,95472,English
...,...,...
100490,158618,Portuguese
100491,158618,Portuguese
100492,158618,Portuguese
100493,158618,Portuguese


## Requisição de Relacionamentos

In [ ]:
userIds = dfs['user']['id'].tolist()

In [ ]:
mediaListQuery = '''
                    mediaList(userId_in:''' + str(userIds) + '''){
                        id
                        media {
                            id
                        }
                        userId
                        status
                        score
                        progress
                        startedAt {
                            day
                            month
                            year
                        }
                    }
                '''

In [ ]:
queries = {
    'mediaList':    (mediaListQuery, 1000), # Só tem 991 entradas ao todo
}

Fazermos a requisição

In [ ]:
for name, (query, total) in queries.items():
    print(f"\n-- {name.capitalize()} --")
    pages = getPages(URL, query, name, total)
    dfs[name] = pd.DataFrame(pages)


-- Medialist --
Dados em cache!


In [ ]:
dfs['mediaList']

,id,media,userId,status,score,progress,startedAt
0,11642079,{'id': 30102},6,PAUSED,80,18,"{'day': None, 'month': None, 'year': None}"
1,11642080,{'id': 30894},6,PAUSED,70,19,"{'day': None, 'month': None, 'year': None}"
2,11642081,{'id': 30107},6,CURRENT,0,48,"{'day': None, 'month': None, 'year': None}"
3,11642082,{'id': 43001},6,CURRENT,90,36,"{'day': None, 'month': None, 'year': None}"
4,11642083,{'id': 30085},6,COMPLETED,100,39,"{'day': None, 'month': None, 'year': None}"
...,...,...,...,...,...,...,...
986,589076546,{'id': 579},24,PLANNING,0,0,"{'day': None, 'month': None, 'year': None}"
987,595250963,{'id': 240},24,PLANNING,0,0,"{'day': None, 'month': None, 'year': None}"
988,595250969,{'id': 1813},24,PLANNING,0,0,"{'day': None, 'month': None, 'year': None}"
989,595251034,{'id': 1288},24,PLANNING,0,0,"{'day': None, 'month': None, 'year': None}"
